# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malaikasaleem944/malaika_flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task

My lane is content refresh prioritization.

**Task type:** Ranking / scoring

The decision is: which content items should an editor fix first?

The model would assign each content item a priority score based on available content, search, traffic, freshness, and performance signals. Higher-priority items would be reviewed first by the content team.

This is a ranking/scoring problem because the goal is not only to predict whether content is declining. The practical goal is to prioritize limited editor time across many content items.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

The target/proxy is a future content-performance outcome that can be used to estimate which content items deserve attention.

For this starter dataset, the available decline information is based on `trend_direction` and `trend_pct`. However, these fields are derived outcome fields and should not be used as input features.

For a future predictive model, the target should be an observed outcome measured after the feature window, such as whether an item's performance declines in a subsequent time window.

The priority score should therefore be based on information available before the outcome window, while the observed future outcome is used to evaluate whether the ranking was useful.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric

The main success metric will be Precision@K.

Precision@K measures how many of the top K items selected by the ranking actually require attention.

This matches the real decision because editors have limited time. The goal is to make the top of the queue useful rather than trying to perfectly rank every content item.

For example, if the team reviews the top 100 content items, Precision@100 tells us what fraction of those selected items were actually associated with the observed future outcome.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### Unit of analysis

**One row = one pseudonymized content item.**

The `content_id` identifies the content item and `client_id` identifies the client associated with it.

The dataset contains 30,000 content-item rows across 32 clients.

The analysis unit is therefore the content item, not the client and not an individual daily performance record.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML beats a fixed rule here

A simple fixed rule could rank content using one signal, such as traffic decline or days since the last update. However, content performance depends on multiple signals at the same time, including impressions, clicks, sessions, search position, content age, freshness, search demand, and content characteristics.

These relationships can be difficult to capture with a small set of manually written rules. ML can learn patterns across multiple signals and produce a priority score that can be updated as more observed outcomes become available.

The output supports a real content action: editors can review the highest-priority content first and decide whether to refresh, improve, or leave an item unchanged.

A fixed rule may still be a useful baseline. ML earns its place only if it improves the ranking quality over that baseline on an honest future outcome.

In [4]:
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "content_age_days",
        "days_since_last_update",
        "avg_position",
        "ctr",
        "trend_direction",
        "trend_pct",
    ]
]

unit_df.head(10)

,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,content_age_days,days_since_last_update,avg_position,ctr,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,17,187,20,10.6,0.76,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,9,445,25,20.3,0.05,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,11,141,20,36.5,0.09,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,78,463,22,6.2,0.49,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,145,263,14,44.0,0.13,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,5,147,20,8.5,0.03,down,-38.9
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,1,90,20,7.0,0.00,down,-92.3
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,28,445,22,21.2,0.06,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,68,90,20,46.0,0.09,down,-58.8
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,3,257,104,4.9,0.16,down,-29.2


## Why ML beats a fixed rule here

A simple fixed rule could rank content using one signal, such as traffic decline or days since the last update. However, content performance depends on multiple signals at the same time, including impressions, clicks, sessions, search position, content age, freshness, search demand, and content characteristics.

These relationships can be difficult to capture with a small set of manually written rules. ML can learn patterns across multiple signals and produce a priority score that can be updated as more observed outcomes become available.

The output supports a real content action: editors can review the highest-priority content first and decide whether to refresh, improve, or leave an item unchanged.

A fixed rule may still be a useful baseline. ML earns its place only if it improves the ranking quality over that baseline on an honest future outcome.

## Self-check

- **Task type:** Ranking / scoring
- **Decision:** Which content should editors fix first?
- **Target/proxy:** An observed future content-performance outcome
- **Success metric:** Precision@K
- **Unit of analysis:** One row per pseudonymized content item
- **Action:** Editors review the highest-priority content items first and decide whether to refresh, improve, or leave them unchanged.
- **Why ML:** Content performance depends on multiple interacting signals, which may be difficult to capture with a small fixed rule.
- **Leakage check:** `trend_direction` and `trend_pct` are outcome-derived and should not be used as model features.
- **Final claim:** ML should only be preferred if it improves ranking quality over a fixed-rule baseline on an honest future outcome.